# 功夫動作分類 - 多種分類模型比較

本 notebook 比較多種分類模型在功夫動作分類任務上的表現：

## 傳統機器學習模型
| 模型 | 描述 |
|------|------|
| Logistic Regression | 邏輯迴歸 - 線性分類器 |
| K-Nearest Neighbors | K近鄰 - 基於距離的分類 |
| Support Vector Machine | 支持向量機 - 尋找最佳超平面 |
| Decision Tree | 決策樹 - 基於規則的分類 |
| Random Forest | 隨機森林 - 決策樹集成 |
| Gradient Boosting | 梯度提升 - 順序集成方法 |
| XGBoost | 極端梯度提升 - 優化的梯度提升 |
| AdaBoost | 自適應提升 - 集成方法 |

## 深度學習模型
| 模型 | 描述 |
|------|------|
| DNN (PyTorch) | 深度神經網路 |

## 1. 匯入套件

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn - 資料處理
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_score, recall_score
)

# Sklearn - 傳統機器學習模型
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier,
    AdaBoostClassifier
)

# XGBoost (如果有安裝)
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
    print("XGBoost 已載入")
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost 未安裝，將跳過")

# PyTorch - 深度學習
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# 檢查 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用裝置: {device}")
print(f"PyTorch 版本: {torch.__version__}")

# 設定隨機種子
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## 2. 載入與預處理資料

In [ ]:
# 讀取資料
csv_path = './dataset/pose_angles_summary_actual.csv'
df = pd.read_csv(csv_path)

print(f"總樣本數: {len(df)}")
print(f"\n各動作類型數量:")
print(df['Action_Type'].value_counts())

# 定義特徵欄位
feature_columns = [
    'R_Elbow_Angle', 'L_Elbow_Angle',
    'R_Knee_Angle', 'L_Knee_Angle',
    'R_Hip_Angle', 'L_Hip_Angle'
]

# 準備特徵和標籤
X = df[feature_columns].values
y = df['Action_Type'].values

# 標籤編碼
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f"\n動作類別對應:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

print(f"\n特徵維度: {X.shape}")
print(f"類別數量: {num_classes}")

In [ ]:
# 分割資料集: 70% 訓練, 15% 驗證, 15% 測試
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.15, random_state=SEED, stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=SEED, stratify=y_temp  # 0.176 ≈ 15/85
)

print(f"訓練集: {len(X_train)} 樣本")
print(f"驗證集: {len(X_val)} 樣本")
print(f"測試集: {len(X_test)} 樣本")

# 標準化特徵
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 3. 定義傳統機器學習模型

In [ ]:
# 定義所有要比較的傳統 ML 模型
ml_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=SEED, multi_class='multinomial'
    ),
    'K-Nearest Neighbors': KNeighborsClassifier(
        n_neighbors=5, weights='distance'
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', random_state=SEED, probability=True
    ),
    'SVM (Linear)': SVC(
        kernel='linear', random_state=SEED, probability=True
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=SEED
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, random_state=SEED
    ),
    'AdaBoost': AdaBoostClassifier(
        n_estimators=100, random_state=SEED, algorithm='SAMME'
    )
}

# 如果有 XGBoost，加入
if HAS_XGBOOST:
    ml_models['XGBoost'] = XGBClassifier(
        n_estimators=100, max_depth=5, random_state=SEED,
        use_label_encoder=False, eval_metric='mlogloss'
    )

print(f"共有 {len(ml_models)} 個傳統 ML 模型要比較")

## 4. 定義深度學習模型 (PyTorch)

In [ ]:
class KungfuDNN(nn.Module):
    """功夫動作分類深度神經網路"""
    
    def __init__(self, input_size=6, num_classes=4, dropout_rate=0.3):
        super(KungfuDNN, self).__init__()
        
        self.network = nn.Sequential(
            # 隱藏層 1
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 隱藏層 2
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 隱藏層 3
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            # 隱藏層 4
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            # 輸出層
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)


def train_dnn(model, train_loader, val_loader, device, num_epochs=100, patience=20):
    """訓練 DNN 模型"""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # 訓練
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # 驗證
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = val_correct / val_total
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, best_val_acc


print("DNN 模型已定義")

## 5. 訓練與評估所有模型

In [ ]:
# 儲存結果
results = {}

print("=" * 70)
print("開始訓練與評估傳統機器學習模型...")
print("=" * 70)

for name, model in ml_models.items():
    start_time = time.time()
    
    # 訓練模型
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start_time
    
    # 預測
    y_pred_train = model.predict(X_train_scaled)
    y_pred_val = model.predict(X_val_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # 計算指標
    train_acc = accuracy_score(y_train, y_pred_train)
    val_acc = accuracy_score(y_val, y_pred_val)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test, average='weighted')
    test_precision = precision_score(y_test, y_pred_test, average='weighted')
    test_recall = recall_score(y_test, y_pred_test, average='weighted')
    
    # 交叉驗證
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    
    results[name] = {
        'type': 'ML',
        'train_time': train_time,
        'train_acc': train_acc,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'predictions': y_pred_test,
        'labels': y_test
    }
    
    print(f"  {name}: 測試準確率 = {test_acc*100:.2f}%, F1 = {test_f1*100:.2f}%, "
          f"訓練時間 = {train_time:.3f}s")

print()

In [ ]:
print("=" * 70)
print("開始訓練與評估深度學習模型 (DNN)...")
print("=" * 70)

# 準備 PyTorch 資料載入器
train_tensor = TensorDataset(
    torch.FloatTensor(X_train_scaled), 
    torch.LongTensor(y_train)
)
val_tensor = TensorDataset(
    torch.FloatTensor(X_val_scaled), 
    torch.LongTensor(y_val)
)
test_tensor = TensorDataset(
    torch.FloatTensor(X_test_scaled), 
    torch.LongTensor(y_test)
)

train_loader = DataLoader(train_tensor, batch_size=16, shuffle=True)
val_loader = DataLoader(val_tensor, batch_size=16, shuffle=False)
test_loader = DataLoader(test_tensor, batch_size=16, shuffle=False)

# 訓練 DNN
start_time = time.time()
dnn_model = KungfuDNN(input_size=6, num_classes=num_classes)
dnn_model, best_val_acc = train_dnn(dnn_model, train_loader, val_loader, device)
train_time = time.time() - start_time

# 評估 DNN
dnn_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features = features.to(device)
        outputs = dnn_model(features)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

y_pred_dnn = np.array(all_preds)
y_true_dnn = np.array(all_labels)

# 計算 DNN 指標
dnn_test_acc = accuracy_score(y_true_dnn, y_pred_dnn)
dnn_test_f1 = f1_score(y_true_dnn, y_pred_dnn, average='weighted')
dnn_test_precision = precision_score(y_true_dnn, y_pred_dnn, average='weighted')
dnn_test_recall = recall_score(y_true_dnn, y_pred_dnn, average='weighted')

results['DNN (PyTorch)'] = {
    'type': 'DL',
    'train_time': train_time,
    'train_acc': None,  # DNN 的訓練準確率需要額外計算
    'val_acc': best_val_acc,
    'test_acc': dnn_test_acc,
    'test_f1': dnn_test_f1,
    'test_precision': dnn_test_precision,
    'test_recall': dnn_test_recall,
    'cv_mean': None,
    'cv_std': None,
    'predictions': y_pred_dnn,
    'labels': y_true_dnn
}

print(f"  DNN (PyTorch): 測試準確率 = {dnn_test_acc*100:.2f}%, F1 = {dnn_test_f1*100:.2f}%, "
      f"訓練時間 = {train_time:.3f}s")

print("\n" + "=" * 70)
print("所有模型訓練完成！")
print("=" * 70)

## 6. 結果比較表格

In [ ]:
# 建立結果表格
results_df = pd.DataFrame({
    '模型': list(results.keys()),
    '類型': [results[k]['type'] for k in results],
    '測試準確率(%)': [round(results[k]['test_acc'] * 100, 2) for k in results],
    'F1 分數(%)': [round(results[k]['test_f1'] * 100, 2) for k in results],
    '精確率(%)': [round(results[k]['test_precision'] * 100, 2) for k in results],
    '召回率(%)': [round(results[k]['test_recall'] * 100, 2) for k in results],
    '驗證準確率(%)': [round(results[k]['val_acc'] * 100, 2) if results[k]['val_acc'] else '-' for k in results],
    '訓練時間(秒)': [round(results[k]['train_time'], 3) for k in results],
    '5-fold CV 平均': [f"{results[k]['cv_mean']*100:.2f}±{results[k]['cv_std']*100:.2f}" 
                      if results[k]['cv_mean'] else '-' for k in results]
})

# 按測試準確率排序
results_df = results_df.sort_values('測試準確率(%)', ascending=False).reset_index(drop=True)

print("\n" + "=" * 100)
print("模型比較結果 (按測試準確率排序)")
print("=" * 100)
print(results_df.to_string(index=False))
print("=" * 100)

## 7. 視覺化比較

In [ ]:
# 準備繪圖資料
model_names = list(results.keys())
test_accs = [results[k]['test_acc'] * 100 for k in model_names]
f1_scores = [results[k]['test_f1'] * 100 for k in model_names]
train_times = [results[k]['train_time'] for k in model_names]
model_types = [results[k]['type'] for k in model_names]

# 設定顏色
colors = ['#FF6B6B' if t == 'DL' else '#4ECDC4' for t in model_types]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 測試準確率比較
ax1 = axes[0, 0]
bars1 = ax1.barh(range(len(model_names)), test_accs, color=colors)
ax1.set_yticks(range(len(model_names)))
ax1.set_yticklabels(model_names)
ax1.set_xlabel('準確率 (%)')
ax1.set_title('測試集準確率比較', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 105)
for i, (bar, acc) in enumerate(zip(bars1, test_accs)):
    ax1.text(acc + 1, bar.get_y() + bar.get_height()/2, f'{acc:.1f}%', 
             va='center', fontsize=9)

# 2. F1 分數比較
ax2 = axes[0, 1]
bars2 = ax2.barh(range(len(model_names)), f1_scores, color=colors)
ax2.set_yticks(range(len(model_names)))
ax2.set_yticklabels(model_names)
ax2.set_xlabel('F1 分數 (%)')
ax2.set_title('測試集 F1 分數比較', fontsize=14, fontweight='bold')
ax2.set_xlim(0, 105)
for i, (bar, f1) in enumerate(zip(bars2, f1_scores)):
    ax2.text(f1 + 1, bar.get_y() + bar.get_height()/2, f'{f1:.1f}%', 
             va='center', fontsize=9)

# 3. 訓練時間比較 (對數尺度)
ax3 = axes[1, 0]
bars3 = ax3.barh(range(len(model_names)), train_times, color=colors)
ax3.set_yticks(range(len(model_names)))
ax3.set_yticklabels(model_names)
ax3.set_xlabel('訓練時間 (秒)')
ax3.set_title('訓練時間比較', fontsize=14, fontweight='bold')
for i, (bar, t) in enumerate(zip(bars3, train_times)):
    ax3.text(t + 0.01, bar.get_y() + bar.get_height()/2, f'{t:.3f}s', 
             va='center', fontsize=9)

# 4. 準確率 vs 訓練時間 散點圖
ax4 = axes[1, 1]
for i, (name, acc, t, c) in enumerate(zip(model_names, test_accs, train_times, colors)):
    ax4.scatter(t, acc, s=200, c=c, alpha=0.7, edgecolors='black', linewidth=1)
    ax4.annotate(name, (t, acc), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax4.set_xlabel('訓練時間 (秒)')
ax4.set_ylabel('測試準確率 (%)')
ax4.set_title('效率分析: 準確率 vs 訓練時間', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

# 添加圖例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4ECDC4', label='傳統 ML'),
    Patch(facecolor='#FF6B6B', label='深度學習')
]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, 
           bbox_to_anchor=(0.5, 0.98), fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('./model/classification_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n比較圖已儲存至 ./model/classification_models_comparison.png")

## 8. 各模型混淆矩陣

In [ ]:
# 選擇前 6 個最佳模型繪製混淆矩陣
sorted_models = sorted(results.keys(), key=lambda k: results[k]['test_acc'], reverse=True)[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, name in enumerate(sorted_models):
    cm = confusion_matrix(results[name]['labels'], results[name]['predictions'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                ax=axes[idx])
    
    model_type = results[name]['type']
    acc = results[name]['test_acc'] * 100
    axes[idx].set_title(f'{name}\nAcc: {acc:.1f}% [{model_type}]', fontsize=10)
    axes[idx].set_xlabel('預測類別')
    axes[idx].set_ylabel('實際類別')

plt.tight_layout()
plt.savefig('./model/confusion_matrices_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

print("混淆矩陣已儲存至 ./model/confusion_matrices_all_models.png")

## 9. 各類別表現分析

In [ ]:
# 選擇最佳模型進行詳細分析
best_model_name = max(results.keys(), key=lambda k: results[k]['test_acc'])
best_result = results[best_model_name]

print(f"\n最佳模型: {best_model_name}")
print(f"測試準確率: {best_result['test_acc']*100:.2f}%")
print(f"\n詳細分類報告:")
print(classification_report(
    best_result['labels'], 
    best_result['predictions'], 
    target_names=label_encoder.classes_
))

In [ ]:
# 各類別在不同模型上的表現
class_metrics = {}

for name, data in results.items():
    report = classification_report(
        data['labels'], data['predictions'], 
        target_names=label_encoder.classes_,
        output_dict=True
    )
    
    for cls in label_encoder.classes_:
        if cls not in class_metrics:
            class_metrics[cls] = {}
        class_metrics[cls][name] = report[cls]['f1-score']

# 繪製各類別在不同模型上的 F1 分數
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_names))
width = 0.2
multiplier = 0

action_names = {
    'act1': '握拳式',
    'act2': '出拳式',
    'act3': '踢腿式',
    'act4': '提膝式'
}

colors_class = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

for i, (cls, metrics) in enumerate(class_metrics.items()):
    offset = width * multiplier
    f1_values = [metrics[name] * 100 for name in model_names]
    bars = ax.bar(x + offset, f1_values, width, label=f'{cls} ({action_names[cls]})', 
                  color=colors_class[i])
    multiplier += 1

ax.set_ylabel('F1 分數 (%)')
ax.set_title('各類別在不同模型上的 F1 分數比較', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, rotation=45, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./model/class_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n各類別表現比較圖已儲存至 ./model/class_performance_comparison.png")

## 10. 模型效能雷達圖

In [ ]:
# 選擇前 5 個最佳模型繪製雷達圖
top_models = sorted(results.keys(), key=lambda k: results[k]['test_acc'], reverse=True)[:5]

# 準備雷達圖資料
categories = ['準確率', 'F1 分數', '精確率', '召回率']
num_vars = len(categories)

# 計算角度
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # 閉合

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

colors_radar = plt.cm.Set2(np.linspace(0, 1, len(top_models)))

for i, name in enumerate(top_models):
    values = [
        results[name]['test_acc'] * 100,
        results[name]['test_f1'] * 100,
        results[name]['test_precision'] * 100,
        results[name]['test_recall'] * 100
    ]
    values += values[:1]  # 閉合
    
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=colors_radar[i])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 105)
ax.set_title('Top 5 模型效能雷達圖', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('./model/model_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("雷達圖已儲存至 ./model/model_radar_chart.png")

## 11. 結論與建議

In [ ]:
# 找出各類別最佳
best_acc_model = max(results.keys(), key=lambda k: results[k]['test_acc'])
best_f1_model = max(results.keys(), key=lambda k: results[k]['test_f1'])
fastest_model = min(results.keys(), key=lambda k: results[k]['train_time'])

# 找出最佳 ML 和 DL 模型
ml_models_results = {k: v for k, v in results.items() if v['type'] == 'ML'}
dl_models_results = {k: v for k, v in results.items() if v['type'] == 'DL'}

best_ml_model = max(ml_models_results.keys(), key=lambda k: ml_models_results[k]['test_acc'])
best_dl_model = max(dl_models_results.keys(), key=lambda k: dl_models_results[k]['test_acc']) if dl_models_results else None

print("\n" + "=" * 70)
print("結論與建議")
print("=" * 70)

print(f"\n🏆 最高準確率模型: {best_acc_model}")
print(f"   測試準確率: {results[best_acc_model]['test_acc']*100:.2f}%")
print(f"   F1 分數: {results[best_acc_model]['test_f1']*100:.2f}%")

print(f"\n🎯 最高 F1 分數模型: {best_f1_model}")
print(f"   F1 分數: {results[best_f1_model]['test_f1']*100:.2f}%")

print(f"\n⚡ 最快訓練模型: {fastest_model}")
print(f"   訓練時間: {results[fastest_model]['train_time']:.4f} 秒")
print(f"   準確率: {results[fastest_model]['test_acc']*100:.2f}%")

print(f"\n📊 最佳傳統 ML 模型: {best_ml_model}")
print(f"   測試準確率: {results[best_ml_model]['test_acc']*100:.2f}%")

if best_dl_model:
    print(f"\n🧠 最佳深度學習模型: {best_dl_model}")
    print(f"   測試準確率: {results[best_dl_model]['test_acc']*100:.2f}%")

print("\n" + "-" * 70)
print("建議:")
print("-" * 70)

# 判斷 ML vs DL
if best_dl_model and results[best_dl_model]['test_acc'] > results[best_ml_model]['test_acc']:
    print(f"\n✅ 深度學習模型 ({best_dl_model}) 表現最佳")
    print(f"   但訓練時間較長 ({results[best_dl_model]['train_time']:.2f}s)")
else:
    print(f"\n✅ 傳統 ML 模型 ({best_ml_model}) 表現最佳")
    print(f"   且訓練速度快 ({results[best_ml_model]['train_time']:.4f}s)")

print(f"\n📌 對於此功夫動作分類任務:")
print(f"   - 若追求最高準確率: 使用 {best_acc_model}")
print(f"   - 若需要快速訓練: 使用 {fastest_model}")
print(f"   - 若需平衡效能與速度: 使用 {best_ml_model}")

print("\n" + "=" * 70)

In [ ]:
# 儲存比較結果
results_df.to_csv('./model/classification_models_comparison.csv', index=False, encoding='utf-8-sig')
print("\n結果已儲存至 ./model/classification_models_comparison.csv")

# 顯示最終結果表格
print("\n最終比較結果:")
display(results_df)

## 12. 總結

### 比較的模型

**傳統機器學習模型:**
- Logistic Regression (邏輯迴歸)
- K-Nearest Neighbors (K近鄰)
- Support Vector Machine (支持向量機) - RBF 核與線性核
- Decision Tree (決策樹)
- Random Forest (隨機森林)
- Gradient Boosting (梯度提升)
- AdaBoost (自適應提升)
- XGBoost (極端梯度提升)

**深度學習模型:**
- DNN (深度神經網路) - PyTorch 實現

### 評估指標
- 測試準確率 (Accuracy)
- F1 分數 (F1-Score)
- 精確率 (Precision)
- 召回率 (Recall)
- 訓練時間
- 5-fold 交叉驗證

### 輸出檔案
- `model/classification_models_comparison.csv` - 比較結果表格
- `model/classification_models_comparison.png` - 模型比較圖
- `model/confusion_matrices_all_models.png` - 混淆矩陣比較
- `model/class_performance_comparison.png` - 各類別表現比較
- `model/model_radar_chart.png` - 模型效能雷達圖